# Metop-SG (Metop Second Generation) / 3MI (Multi-viewing, Multi-channel, Multi-polarisation Imaging)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=metop-sg/metop-sg-3mi). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1B

Level 1B: calibrated, multi-angle (up to 14 views) polarised radiances with geolocation.

Looks for 3MI L1B collections in the EUMETSAT Data Store (title search, prints the matches) and downloads the first product of the window. Free account and consumer key/secret needed; the collection may not be public yet.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collections = [c for c in datastore.collections if "3mi" in str(c.title).lower()]
print([(c.collection_id, str(c.title)) for c in collections])
if not collections:
    raise SystemExit("Collection not found in the Data Store yet - check data.eumetsat.int")
collection = collections[0]

products = list(collection.search(dtstart=dt.datetime(2026, 6, 1, 12, 0), dtend=dt.datetime(2026, 6, 1, 13, 0)))
print(len(products), "3MI L1B products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

import xarray as xr

files = sorted(out.rglob("*.nc"))
print(files[:3])
if files:
    try:
        ds = xr.open_dataset(files[0], engine="h5netcdf")
        print(ds)
    except Exception as exc:                      # some products need group="..." or another engine
        print("Could not open directly:", exc)
else:
    print("No .nc file: the product is in EPS native / BUFR / HDF5 format - read it with satpy, eccodes or the EUMETSAT tools")

## L2

Level 2: aerosol optical depth, aerosol type and cloud properties from multi-angle polarimetry.

Looks for 3MI L2 collections in the EUMETSAT Data Store (title search, prints the matches) and downloads the first product of the window. Free account and consumer key/secret needed; the collection may not be public yet.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collections = [c for c in datastore.collections if "3mi" in str(c.title).lower()]
print([(c.collection_id, str(c.title)) for c in collections])
if not collections:
    raise SystemExit("Collection not found in the Data Store yet - check data.eumetsat.int")
collection = collections[0]

products = list(collection.search(dtstart=dt.datetime(2026, 6, 1, 12, 0), dtend=dt.datetime(2026, 6, 1, 13, 0)))
print(len(products), "3MI L2 products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

import xarray as xr

files = sorted(out.rglob("*.nc"))
print(files[:3])
if files:
    try:
        ds = xr.open_dataset(files[0], engine="h5netcdf")
        print(ds)
    except Exception as exc:                      # some products need group="..." or another engine
        print("Could not open directly:", exc)
else:
    print("No .nc file: the product is in EPS native / BUFR / HDF5 format - read it with satpy, eccodes or the EUMETSAT tools")